# Capítulo 3 — Amostragem e Estimação

Notebook com o **código** deste capítulo, para o Google Colab. Cada trecho vem precedido de uma breve explicação; o texto completo está no site do livro.

Rode a célula de **setup** abaixo primeiro (uma vez), depois as demais em ordem.

In [ ]:
# Setup (rode uma vez).
!curl -sO https://raw.githubusercontent.com/BragaD/UnDF-Bases3-Estatistica-202602/main/formato.py   # baixa o ajudante de formatação do livro

## 3.1 — Amostragem Aleatória e Viés de Amostra

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from formato import num

plt.rcParams["figure.figsize"] = (7, 4)
renda = pd.read_csv("https://raw.githubusercontent.com/BragaD/UnDF-Bases3-Estatistica-202602/main/dados/loans_income.csv").rename(columns={"x": "Renda"})["Renda"]

Para ilustrar amostragem aleatória simples, vamos usar `loans_income.csv`: a renda anual de 50.000 pessoas que solicitaram empréstimo, em **dólares americanos** — o mesmo `loans_income` que a seção 2.5 já usou para o QQ-plot da normal. É um dado americano do livro-texto — diferente do `estados.csv` do Capítulo 1, que é brasileiro —, e vai reaparecer nas próximas seções deste capítulo (distribuição amostral, bootstrap, intervalos de confiança). A coluna chega do CSV como `x`; nós a renomeamos para `Renda` na leitura, sem tocar no arquivo original.

In [ ]:
print(f"População: {num(len(renda), 0)} rendas")
print(f"Média:   {num(renda.mean(), 0)}")
print(f"Mediana: {num(renda.median(), 0)}")

Uma escolha de projeto entra aqui: sortear **com** ou **sem reposição**. Com reposição, cada indivíduo sorteado volta para o conjunto e pode sair de novo; sem reposição, ele sai da lista depois de escolhido. Na prática, a diferença quase nunca importa: com 50.000 rendas na população e 100 na amostra, a chance de o mesmo indivíduo ser sorteado duas vezes é próxima de zero, com ou sem reposição. A diferença só passa a valer quando a amostra é **grande em relação à população** — sortear 40 de uma turma de 50, por exemplo, onde cada retirada muda visivelmente quem resta para a próxima.

In [ ]:
amostra = renda.sample(100, random_state=42)
print(f"Média da amostra (n=100): {num(amostra.mean(), 0)}")
print(f"Média da população:       {num(renda.mean(), 0)}")

## 3.2 — Viés de Seleção

In [ ]:
import numpy as np
from formato import num

Data snooping é o hábito; o **efeito da busca vasta** (*vast search effect*) é o mecanismo por trás dele, e vale a pena isolar em uma simulação. A ideia central: **procurar em hipóteses suficientes garante achar uma que parece impressionante por puro acaso** — nenhuma fraude, nenhum erro de cálculo, só o número de tentativas fazendo o trabalho.

In [ ]:
rng = np.random.default_rng(42)

# 100 pessoas jogam uma moeda JUSTA 20 vezes cada. Quantas caras a "melhor" tira?
tentativas = [int(rng.integers(0, 2, 20).sum()) for _ in range(100)]
melhor = max(tentativas)
print(f"Em 100 tentativas de 20 lançamentos de uma moeda justa,")
print(f"a melhor sequência deu {melhor} caras de 20.")

## 3.3 — Distribuição Amostral de uma Estatística

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from formato import num

plt.rcParams["figure.figsize"] = (7, 4)
renda = pd.read_csv("https://raw.githubusercontent.com/BragaD/UnDF-Bases3-Estatistica-202602/main/dados/loans_income.csv").rename(columns={"x": "Renda"})["Renda"]

A **distribuição amostral de uma estatística** é outra distribuição inteiramente: pegue a *média* de uma amostra, anote-a, sorteie outra amostra do mesmo tamanho, tire a média de novo, anote de novo — repita isso muitas vezes. O conjunto dessas médias tem sua própria distribuição, com sua própria forma, seu próprio centro e seu próprio espalhamento. Ela não descreve rendas individuais; descreve como a *média de uma amostra* varia de amostra para amostra.

In [ ]:
rng_medias = lambda n: [renda.sample(n, random_state=i).mean() for i in range(1000)]

fig, eixos = plt.subplots(1, 3, figsize=(11, 3.3), sharex=True)
for ax, (n, dados, titulo) in zip(eixos, [
    (1, renda.sample(1000, random_state=0), "Dados (n=1)"),
    (5, rng_medias(5), "Médias de n=5"),
    (20, rng_medias(20), "Médias de n=20"),
]):
    ax.hist(dados, bins=40, color="#b0c4d8", edgecolor="white")
    ax.set_title(titulo, fontsize=10)
    ax.set_xlabel("Renda (US$)")
eixos[0].set_ylabel("Frequência")
plt.tight_layout()
plt.show()

onde $\sigma$ é o desvio-padrão da população e $n$ é o tamanho de cada amostra. A fórmula não é uma aproximação a ser tomada de fé — dá para verificar diretamente, simulando o processo de amostragem muitas vezes e comparando o desvio-padrão empírico das médias resultantes com o que a fórmula prevê.

In [ ]:
dp_pop = renda.std(ddof=1)
print(f"{'n':>4}  {'desvio das médias':>18}  {'σ/√n (teórico)':>16}")
for n in [1, 5, 20, 100]:
    medias = [renda.sample(n, random_state=i).mean() for i in range(1000)]
    print(f"{n:>4}  {num(np.std(medias, ddof=1), 0):>18}  {num(dp_pop / np.sqrt(n), 0):>16}")

## 3.4 — Bootstrap

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from formato import num

plt.rcParams["figure.figsize"] = (7, 4)
renda = pd.read_csv("https://raw.githubusercontent.com/BragaD/UnDF-Bases3-Estatistica-202602/main/dados/loans_income.csv").rename(columns={"x": "Renda"})["Renda"]

Para encenar a situação real — uma amostra, e só ela —, vamos primeiro coletar uma: 1.000 rendas sorteadas da população. Daqui em diante, fingimos que as 50.000 não existem; tudo o que temos é a amostra de 1.000.

In [ ]:
amostra = renda.sample(1000, random_state=1)

n_boot = 1000
medianas = [amostra.sample(len(amostra), replace=True, random_state=i).median()
            for i in range(n_boot)]

print(f"Mediana da amostra:      {num(amostra.median(), 0)}")
print(f"Erro padrão (bootstrap): {num(np.std(medianas, ddof=1), 2)}")

A mediana da nossa amostra é 63.000 — a mediana verdadeira da população é 62.000, mas quem só tem a amostra não sabe disso. O bootstrap não corrige essa distância: ele mede o quanto a mediana *da amostra* oscila, não o quanto ela erra o alvo real. É a diferença entre precisão e acurácia, e voltaremos a ela ao final da seção.

In [ ]:
fig, ax = plt.subplots()
ax.hist(medianas, bins=30, color="#b0c4d8", edgecolor="white")
ax.axvline(amostra.median(), color="#c0392b", linewidth=2, label=f"Mediana da amostra: {num(amostra.median(), 0)}")
ax.set_xlabel("Mediana reamostrada (US$)")
ax.set_ylabel("Frequência")
ax.legend()
plt.tight_layout()
plt.show()

## 3.5 — Intervalos de Confiança

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from formato import num

plt.rcParams["figure.figsize"] = (7, 4)
renda = pd.read_csv("https://raw.githubusercontent.com/BragaD/UnDF-Bases3-Estatistica-202602/main/dados/loans_income.csv").rename(columns={"x": "Renda"})["Renda"]

Suponha que a única informação disponível seja uma amostra pequena: 20 rendas, não as 50.000 da população inteira. É a situação real de quem faz uma pesquisa — coletar mais é caro, e a coleta já aconteceu.

In [ ]:
amostra20 = renda.sample(20, random_state=3)
print(f"Média da amostra (n=20): {num(amostra20.mean(), 0)}")

boot = [amostra20.sample(20, replace=True, random_state=i).mean() for i in range(1000)]
lo, hi = np.percentile(boot, [5, 95])
print(f"IC de 90%: [{num(lo, 0)}, {num(hi, 0)}]")
print(f"Largura: {num(hi - lo, 0)}")

O procedimento é o mesmo bootstrap da seção anterior: reamostrar `amostra20` com reposição, mil vezes, e calcular a média de cada reamostra. A diferença está no que se faz com a lista de 1.000 médias resultante. Em vez de resumi-la a um único número (o desvio-padrão, que dá o erro padrão), pegam-se os **percentis 5 e 95** dessa distribuição: os dois valores que deixam 5% das médias reamostradas abaixo e 5% acima, sobrando exatamente os 90% centrais entre eles. Esse par de valores — aqui, aproximadamente [42.334, 69.241] — é o **intervalo de confiança de 90%** para a renda média da população.

In [ ]:
fig, ax = plt.subplots()
ax.hist(boot, bins=30, color="#b0c4d8", edgecolor="white")
ax.axvline(lo, color="#c0392b", linewidth=2, linestyle="--", label=f"IC 90%: [{num(lo, 0)}, {num(hi, 0)}]")
ax.axvline(hi, color="#c0392b", linewidth=2, linestyle="--")
ax.axvline(amostra20.mean(), color="#27ae60", linewidth=2, label=f"Média da amostra: {num(amostra20.mean(), 0)}")
ax.set_xlabel("Média reamostrada (US$)")
ax.set_ylabel("Frequência")
ax.legend()
plt.tight_layout()
plt.show()

Existe uma segunda escolha embutida em "IC de 90%": por que 90%, e não 95% ou 99%? A resposta é que mais confiança não vem de graça.

In [ ]:
lo95, hi95 = np.percentile(boot, [2.5, 97.5])
print(f"IC de 90%: largura {num(hi - lo, 0)}")
print(f"IC de 95%: largura {num(hi95 - lo95, 0)}")